In [8]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel, AutoImageProcessor, ViTModel
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import itertools
import time
from PIL import Image

In [9]:


# ============================================
# DATA LOADING FUNCTIONS
# ============================================

def load_text_data(folder):
    """Load text files and return texts, labels, and filenames"""
    texts, labels, filenames = [], [], []
    
    # Check if folder exists
    if not os.path.exists(folder):
        print(f"Warning: {folder} does not exist")
        return [], np.array([]), []
    
    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)
        
        if not os.path.exists(subfolder):
            print(f"Warning: {subfolder} does not exist")
            continue
            
        files = sorted([file for file in os.listdir(subfolder) if file.endswith(".txt")])
        for file in tqdm(files, desc=f"Loading {folder}/{label_name}", unit="file", leave=False):
            with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                texts.append(f.read())
            labels.append(label)
            filenames.append((label_name, file))
    
    return texts, np.array(labels), filenames


def load_image_paths(image_folder, filenames):
    """Load image paths corresponding to text files"""
    image_paths = []
    
    for label_name, file in tqdm(filenames, desc=f"Matching images in {image_folder}", unit="img", leave=False):
        img_name = file.replace(".txt", ".png")
        img_path = os.path.join(image_folder, label_name, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Missing image: {img_path}")
            # Try alternative extensions
            for ext in ['.jpg', '.jpeg']:
                alt_path = img_path.replace('.png', ext)
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break
            else:
                raise FileNotFoundError(f"Missing image: {img_path}")
        
        image_paths.append(img_path)
    
    return image_paths

In [10]:



# ============================================
# FEATURE EXTRACTION
# ============================================

def extract_stylometric_features(code: str):
    """Extract 4 stylometric features"""
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    
    return np.array([
        avg_line_length,
        len(lines),
        len(code.split()),
        len(code)
    ], dtype=np.float32)


def compute_stylo(texts):
    """Compute stylometric features for all texts"""
    if len(texts) == 0:
        return np.array([]).reshape(0, 4)
    return np.array([extract_stylometric_features(t) for t in texts])

In [11]:



# ============================================
# DATASET CLASS
# ============================================

class MultiModalDataset(Dataset):
    def __init__(self, texts, images, labels, extra, tokenizer, processor, max_length=256):
        self.texts = texts
        self.images = images
        self.labels = labels
        self.extra = extra
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Load and process image
        img = Image.open(self.images[idx]).convert("RGB")
        
        # Tokenize text
        text_enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Process image
        img_enc = self.processor(images=img, return_tensors="pt")
        
        return {
            "input_ids": text_enc["input_ids"].squeeze(0),
            "attention_mask": text_enc["attention_mask"].squeeze(0),
            "pixel_values": img_enc["pixel_values"].squeeze(0),
            "extra": torch.tensor(self.extra[idx], dtype=torch.float32) if self.extra is not None and len(self.extra) > 0 else torch.zeros(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [12]:
# ============================================
# MODEL ARCHITECTURE
# ============================================

class HybridModel(nn.Module):
    def __init__(self, use_cb, use_vit, extra_dim):
        super().__init__()
        
        self.use_cb = use_cb
        self.use_vit = use_vit
        
        # Initialize models only if needed
        if use_cb:
            self.codet5p = T5EncoderModel.from_pretrained("Salesforce/codet5p-220m")
            # Freeze CodeT5+
            # for param in self.codet5p.parameters():
            #     param.requires_grad = False
        
        if use_vit:
            self.vit = ViTModel.from_pretrained("facebook/deit-base-patch16-224")
            # Freeze ViT
            # for param in self.vit.parameters():
            #     param.requires_grad = False
        
        # Calculate input dimension
        input_dim = 0
        if use_cb: 
            input_dim += 768
        if use_vit: 
            input_dim += 768
        input_dim += extra_dim
        
        self.norm = nn.LayerNorm(input_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    
    def forward(self, batch):
        feats = []
        
        if self.use_cb:
            # with torch.no_grad():  # Freeze CodeT5+
            out = self.codet5p(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                return_dict=True,
            )
            mask = batch["attention_mask"].unsqueeze(-1).to(out.last_hidden_state.dtype)
            pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            feats.append(pooled)
        
        if self.use_vit:
            # with torch.no_grad():  # Freeze ViT
            out = self.vit(pixel_values=batch["pixel_values"])
            feats.append(out.last_hidden_state[:, 0, :])
        
        # Add extra features if they exist
        if batch["extra"].shape[1] > 0:
            feats.append(batch["extra"])
        
        # Concatenate all features
        x = torch.cat(feats, dim=1)
        x = self.norm(x)
        
        return self.classifier(x)

In [13]:
# ============================================
# TRAINING AND EVALUATION
# ============================================

def format_duration(seconds):
    seconds = int(max(seconds, 0))
    hours, rem = divmod(seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def train_and_eval(combo, train_texts, train_labels, train_imgs, train_tfidf, train_metrics, 
                   vectorizer, scaler, device, tokenizer, processor):
    
    combo_start = time.time()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    use_cb = "codet5p" in combo
    use_vit = "vit" in combo
    combo_name = "+".join(combo)
    
    # Build extra features
    extra_list = []
    if "stylometric" in combo and len(train_texts) > 0:
        extra_list.append(compute_stylo(train_texts))
    if "metrics" in combo and train_metrics is not None:
        extra_list.append(train_metrics)
    if "tfidf" in combo and train_tfidf is not None:
        extra_list.append(train_tfidf)
    
    if extra_list:
        extra = np.concatenate(extra_list, axis=1)
        # Scale features
        scaler.fit(extra)
        extra = scaler.transform(extra)
    else:
        extra = np.zeros((len(train_texts), 0))
    
    # Create dataset and dataloader
    dataset = MultiModalDataset(train_texts, train_imgs, train_labels, extra, tokenizer, processor)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)  # Reduced batch size for memory
    
    # Initialize model
    model = HybridModel(use_cb, use_vit, extra.shape[1]).to(device)
    
    # Only train the classifier head (feature extractors are frozen)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()
    
    # Training loop (3 epochs)
    max_epochs = 3
    for epoch in range(max_epochs):
        epoch_start = time.time()
        model.train()
        total_loss = 0
        train_pbar = tqdm(
            loader,
            desc=f"{combo_name} epoch {epoch + 1}/{max_epochs}",
            unit="batch",
            leave=False,
        )
        
        for batch_idx, batch in enumerate(train_pbar, start=1):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(batch)
            loss = loss_fn(outputs, batch["label"])
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            avg_loss_so_far = total_loss / batch_idx
            epoch_elapsed = time.time() - epoch_start
            epoch_eta = (epoch_elapsed / batch_idx) * (len(loader) - batch_idx) if batch_idx else 0
            train_pbar.set_postfix_str(
                f"loss {avg_loss_so_far:.4f}, epoch eta {format_duration(epoch_eta)}, "
                f"combo elapsed {format_duration(time.time() - combo_start)}"
            )
        
        avg_loss = total_loss / max(len(loader), 1)
        print(
            f"  Epoch {epoch+1}/{max_epochs} - Loss: {avg_loss:.4f} "
            f"- Epoch time: {format_duration(time.time() - epoch_start)} "
            f"- Combo elapsed: {format_duration(time.time() - combo_start)}"
        )
    
    # Test on all 10 test sets
    accs = []
    test_sets = range(10)
    test_sets_pbar = tqdm(test_sets, desc=f"{combo_name} test sets", unit="set", leave=False)
    
    for i in test_sets_pbar:
        test_set_start = time.time()
        test_sets_pbar.set_postfix_str(f"loading Test_{i}")
        test_texts, test_labels, test_files = load_text_data(f"../Text_Files/Test_{i}")
        
        if len(test_texts) == 0:
            print(f"Test_{i}: No data found, skipping")
            continue
        
        test_imgs = load_image_paths(f"../snapshots/Test_{i}", test_files)
        
        # Build test extra features
        test_extra_list = []
        if "stylometric" in combo:
            test_extra_list.append(compute_stylo(test_texts))
        if "metrics" in combo:
            metrics_path = f"metrics_test_{i}.npz"
            if os.path.exists(metrics_path):
                test_extra_list.append(np.load(metrics_path)["test_metrics"])
            else:
                print(f"Warning: {metrics_path} not found")
        if "tfidf" in combo:
            test_extra_list.append(vectorizer.transform(test_texts).toarray())
        
        if test_extra_list:
            test_extra = np.concatenate(test_extra_list, axis=1)
            test_extra = scaler.transform(test_extra)
        else:
            test_extra = np.zeros((len(test_texts), 0))
        
        # Create test dataset
        test_dataset = MultiModalDataset(test_texts, test_imgs, test_labels, test_extra, tokenizer, processor)
        test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
        
        # Evaluate
        model.eval()
        preds, ys = [], []
        test_batch_pbar = tqdm(
            test_loader,
            desc=f"{combo_name} Test_{i}",
            unit="batch",
            leave=False,
        )
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(test_batch_pbar, start=1):
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(batch)
                preds.extend(outputs.argmax(dim=1).cpu().numpy())
                ys.extend(batch["label"].cpu().numpy())
                set_elapsed = time.time() - test_set_start
                set_eta = (set_elapsed / batch_idx) * (len(test_loader) - batch_idx) if batch_idx else 0
                test_batch_pbar.set_postfix_str(f"eta {format_duration(set_eta)}")
        
        acc = accuracy_score(ys, preds)
        accs.append(acc)
        test_sets_pbar.set_postfix_str(
            f"Test_{i} acc {acc:.4f}, elapsed {format_duration(time.time() - test_set_start)}"
        )
        print(f"  Test_{i}: {acc:.4f} - Time: {format_duration(time.time() - test_set_start)}")
    
    combo_elapsed = time.time() - combo_start
    print(f"  Combination finished in {format_duration(combo_elapsed)}")
    
    # Cleanup
    del model, optimizer, dataset, loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return np.mean(accs) if accs else 0.0


In [14]:

# ============================================
# MAIN EXECUTION
# ============================================


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and processor
print("Loading tokenizer and processor...")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")
processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load training data
print("Loading training data...")
train_texts, train_labels, train_files = load_text_data("../Text_Files/Train")

if len(train_texts) == 0:
    print("Error: No training data found!")
    exit(1)

print(f"Loaded {len(train_texts)} training samples")

# Load training images
train_imgs = load_image_paths("../snapshots/Train", train_files)
print(f"Loaded {len(train_imgs)} training images")

# Compute TF-IDF features
print("Computing TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=500)
train_tfidf = vectorizer.fit_transform(train_texts).toarray()

# Load pre-computed metrics (if available)
train_metrics = None
if os.path.exists("metrics_data.npz"):
    train_metrics = np.load("metrics_data.npz")["train_metrics"]
    print(f"Loaded metrics with shape: {train_metrics.shape}")
else:
    print("Warning: metrics_data.npz not found")
train_metrics = train_metrics[:, 1:6]
# Initialize scaler
scaler = StandardScaler()

# Feature combinations to try
features = ["codet5p", "vit", "stylometric", "tfidf", "metrics"]

best_acc = 0
best_combo = None

# Try all combinations
all_combos = [combo for r in range(1, len(features) + 1) for combo in itertools.combinations(features, r)]
sweep_start = time.time()

with tqdm(total=len(all_combos), desc="All feature combinations", unit="combo", leave=True) as sweep_pbar:
    for combo_index, combo in enumerate(all_combos, start=1):
        combo_start = time.time()
        print(f"\n{'='*50}")
        print(f"Testing combination {combo_index}/{len(all_combos)}: {combo}")
        print(f"{'='*50}")
        
        try:
            acc = train_and_eval(combo, train_texts, train_labels, train_imgs, 
                                train_tfidf, train_metrics, vectorizer, scaler, 
                                device, tokenizer, processor)
            print(f"\n>>> AVG Accuracy for {combo}: {acc:.4f} <<<")
            print(f">>> Combination runtime: {format_duration(time.time() - combo_start)} <<<\n")
            
            if acc > best_acc:
                best_acc = acc
                best_combo = combo
        except Exception as e:
            print(f"Error with combination {combo}: {e}")
        finally:
            sweep_pbar.update(1)
            elapsed = time.time() - sweep_start
            remaining = len(all_combos) - sweep_pbar.n
            eta = (elapsed / sweep_pbar.n) * remaining if sweep_pbar.n else 0
            sweep_pbar.set_postfix_str(
                f"elapsed {format_duration(elapsed)}, eta {format_duration(eta)}, "
                f"best {best_acc:.4f}"
            )

print("\n" + "="*50)
print(f"BEST COMBINATION: {best_combo}")
print(f"BEST ACCURACY: {best_acc:.4f}")
print(f"TOTAL SWEEP TIME: {format_duration(time.time() - sweep_start)}")
print("="*50)


Using device: cuda
Loading tokenizer and processor...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading training data...


Loaded 6190 training samples


Loaded 6190 training images
Computing TF-IDF features...


...

  Test_8: 0.5060 - Time: 5s















































































































































































































































































































All feature combinations:  16%|█▌        | 5/31 [26:00<1:49:18, 252.27s/combo, elapsed 26m 00s, eta 2h 15m 13s, best 0.7699]

  Test_9: 0.5030 - Time: 5s
  Combination finished in 3m 06s

>>> AVG Accuracy for ('metrics',): 0.5050 <<<
>>> Combination runtime: 3m 06s <<<


Testing combination 6/31: ('codet5p', 'vit')


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
All feature combinations:  19%|█▉        | 6/31 [27:33<1:54:50, 275.64s/combo, elapsed 27m 33s, eta 1h 54m 50s, best 0.7699]


KeyboardInterrupt: 